In [1]:
# ============================================================================
# IMPORTS
# ============================================================================
# All imports consolidated here for the block synchronization pipeline

from __future__ import annotations
from pathlib import Path
from typing import Optional, Tuple, Literal, Sequence, Union
from dataclasses import dataclass
import re
import pickle

# Core data science
import numpy as np
import pandas as pd

# Bokeh for interactive visualization
from bokeh.io import output_notebook, show, reset_output
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, CustomJS, Span, HoverTool
from bokeh.palettes import Category10
try:
    # Bokeh 3.x
    from bokeh.models import Slider
    from bokeh.resources import INLINE
except Exception:
    # Bokeh 2.x
    from bokeh.models.widgets import Slider
    from bokeh.resources import INLINE
from bokeh.layouts import column, row

# Project utilities
from eye_tracking_system_tools.preprocessing import utility_functions as uf

# Optional visualization (for jitter analysis plots)
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# ============================================================================
# Block synchronization pipeline — all logic lives in block_sync_core and block_sync_visualization
# ============================================================================

from eye_tracking_system_tools.preprocessing.block_sync_core import (
    simple_sync_build,
    shift_eye_df_by_index,
    build_final_sync_df_merge_nearest,
    verify_final_df_against_sources,
    export_final_sync_df,
    load_final_sync_df,
    build_arena_grid_df,
    ArenaGridInfo,
    create_distance_plot,
    add_intermediate_elements,
    find_jittery_frames,
    export_eye_data_2d,
    describe_eye_tick,
)

from eye_tracking_system_tools.preprocessing.block_sync_visualization import (
    plot_simple_sync_bokeh,
    hover_inspect_eyes_bokeh,
    sanity_plot_final_df,
    insert_dup_by_pos,
    insert_dup_by_oe_sample,
    insert_duplicate_frames_slide,
    remove_frame_at_pos,
    interactive_sync_tool_bokeh,
    plot_sync_verification_with_electrophys,
    plot_led_off_events_viewer,
)


# Block Synchronization Pipeline

This notebook implements the complete block synchronization workflow, bringing eye tracking data from raw videos to synchronized `left_eye_data` and `right_eye_data` DataFrames ready for downstream analysis.

## Workflow Overview

1. **Setup**: Initialize BlockSync object
2. **Data Preparation**: Handle videos, parse Open Ephys events, extract brightness
3. **Arena Grid**: Build 60Hz master grid from arena TTL events
4. **Simple Synchronization**: Initial eye sync using first TTL anchor
5. **Manual Correction**: Interactive alignment with LED events
6. **Final Merge**: Merge corrected eyes onto arena grid
7. **Verification**: Check alignment quality
8. **Jitter Correction**: Remove camera jitter artifacts
9. **LED Blink Removal**: Remove LED blink artifacts
10. **Final Export**: Create left/right_eye_data for downstream use

---

## Step 1: Block Setup and Initialization

In [3]:
# block instantiation:
bad_blocks = [] #
experiment_path = Path(r"D:\sample_data_for_eye_repo")

block_numbers = [6]
animal = 'PV_126'
block_collection = uf.block_generator(block_numbers=block_numbers,
                                      experiment_path=experiment_path,
                                      animal=animal,
                                      bad_blocks=bad_blocks,regev=True,
                                      )
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
    elif block.animal_call == "TE_21":
        block.channeldict={1:'Arena_TTL',
                           4:'LED_driver',
                           5:'R_eye_TTL',
                           8:'L_eye_TTL'}
    elif block.animal_call == "PV_106":
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
# create a block_dict object for ease of access:
block_dict = {}
for b in block_collection:
    block_dict[str(b.block_num)] = b

instantiated block number 006 at Path: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006, new OE version
Found the sample rate for block 006 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\oe_files\PV126_Trial15_hunter7_2024-07-18_12-25-35\Record Node 102...
xml data matches file data.

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode)
retrieving zertoh sample number for block 006
got it!


## Step 2: Data Preparation

Run the following methods to prepare the block data:
- `handle_eye_videos()`: Convert and validate eye video files
- `parse_open_ephys_events()`: Parse Open Ephys events (includes manual TTL selector for non-standard paradigms)
- `handle_arena_files()`: Process arena video files
- `get_eye_brightness_vectors()`: Extract brightness values from eye videos

In [4]:
block = block_collection[0]
block.handle_eye_videos()
block.parse_open_ephys_events()
block.handle_arena_files()
block.get_eye_brightness_vectors()


handling eye video files
converting videos...
converting files: ['D:\\sample_data_for_eye_repo\\PV_126\\2024_07_18\\block_006\\eye_videos\\LE\\hunter7_640x480_60hz_experiment_1_recording_0\\hunter7.h264', 'D:\\sample_data_for_eye_repo\\PV_126\\2024_07_18\\block_006\\eye_videos\\RE\\hunter7_640x480_60hz_experiment_1_recording_0\\hunter7.h264'] 
 avoiding conversion on files: ['D:\\sample_data_for_eye_repo\\PV_126\\2024_07_18\\block_006\\eye_videos\\LE\\hunter7_640x480_60hz_experiment_1_recording_0\\hunter7_LE.mp4', 'D:\\sample_data_for_eye_repo\\PV_126\\2024_07_18\\block_006\\eye_videos\\RE\\hunter7_640x480_60hz_experiment_1_recording_0\\hunter7.mp4']
The file D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\eye_videos\RE\hunter7_640x480_60hz_experiment_1_recording_0\hunter7.mp4 already exists, no conversion necessary
Validating videos...
The video named hunter7_LE.mp4 has reported 79224 frames and has 79224 frames, it has dropped 0 frames
The video named hunter7.mp4 has reported

Processing D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\eye_videos\LE\hunter7_640x480_60hz_experiment_1_recording_0\hunter7_LE.mp4: 100%|██████████| 79224/79224 [10:41<00:00, 123.49frame/s]


Finished video D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\eye_videos\LE\hunter7_640x480_60hz_experiment_1_recording_0\hunter7_LE.mp4, processed 79224 frames
Working on video D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\eye_videos\RE\hunter7_640x480_60hz_experiment_1_recording_0\hunter7.mp4


Processing D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\eye_videos\RE\hunter7_640x480_60hz_experiment_1_recording_0\hunter7.mp4: 100%|██████████| 79217/79217 [10:40<00:00, 123.71frame/s]


Finished video D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\eye_videos\RE\hunter7_640x480_60hz_experiment_1_recording_0\hunter7.mp4, processed 79217 frames
Eye brightness vectors generation complete.


## Step 3: Build Arena Grid

Create the master 60Hz grid from arena TTL events. This grid will be used to align all streams. The function automatically handles cases where arena fps differs from 60Hz by creating a pseudo-60Hz grid.

In [6]:
# after block.parse_open_ephys_events() succeeded (manual or auto)
arena_grid_df, info = build_arena_grid_df(block, target_fps=60.0, arena_fps_tol_hz=5.0)

# info.used_pseudo_60hz tells you whether it applied the correction
print(info)

[arena] inferred fps ≈ 30.488 Hz (median step 656 samples @ fs=20000.0 Hz)
[arena] arena fps not ~60.0. Building pseudo 60.0Hz grid over window.
[arena] grid rows=78,866 | unique source arena frames mapped=40,005
[arena] NOTE: Arena_frame_src will repeat (upsampled arena). Use Arena_frame_60 as the 60Hz master frame index.
ArenaGridInfo(fs_hz=20000.0, inferred_arena_fps=30.48780487804878, inferred_arena_step_samp=656, target_fps=60.0, used_pseudo_60hz=True, start_samp=411321, end_samp=26673486, n_grid=78866)


## Step 4: Simple Eye Synchronization

Build per-eye DataFrames using the simple anchor-at-first-TTL approach. This creates initial synchronization that can be manually corrected in the next step.

The algorithm:
1. Read internal timestamps for each eye
2. Get the FIRST TTL sample for that eye
3. Place frame 0 at that TTL, others by internal timing deltas
4. Attach brightness values

In [7]:
# Build and save the per-eye DataFrames
dfL, dfR = simple_sync_build(block, export=True)

# Quick look and manual correction to LED grid
plot_simple_sync_bokeh(block, dfL, dfR, show_led=True)


[LEFT] frames=79,224 | median fps=60.107 | CoV(dt)=0.03% | outliers(±1–99%)=1.91%
[RIGHT] frames=79,217 | median fps=60.107 | CoV(dt)=0.01% | outliers(±1–99%)=1.47%
[OK] Saved: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\analysis\eye_left_simple_sync.csv
[OK] Saved: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\analysis\eye_right_simple_sync.csv
[INFO] Slider tick ≈ 16.650 ms (Left), 16.650 ms (Right)


In [8]:
# Inspect ms per tick (helps translate slider “ticks” to ms)
print("Left tick ≈ %.3f ms"  % describe_eye_tick(dfL))
print("Right tick ≈ %.3f ms" % describe_eye_tick(dfR))

Left tick ≈ 16.650 ms
Right tick ≈ 16.650 ms


## Step 5: Manual Synchronization Correction

Use the interactive Bokeh plot to visually align the eye traces with LED events. Adjust sliders to shift traces, then apply corrections using `shift_eye_df_by_index()`.

**Note**: The plot opens in your default browser. Use the sliders to find the correct shift values, then apply them in the next cell.

In [9]:
# Use the function to apply shift, 
# Notice: the values here should be inverse to the shift which corrects the interactive plot
dfL_shifted = shift_eye_df_by_index(dfL,-1)
dfR_shifted = shift_eye_df_by_index(dfR,-1)

## Step 6: Verification & optional drift-shift

Verify the final synchronization by plotting.
If required, optional dropped frame correction approach is available via frame duplication and dataframe shifts

In [10]:
# use this function to verify shift implemented
plot_simple_sync_bokeh(block, dfL_shifted, dfR_shifted, show_led=True)

[INFO] Slider tick ≈ 16.650 ms (Left), 16.650 ms (Right)


### Optional: Interactive sync correction (duplicate/remove frames)

If eye data has shifted forward relative to OE events (temporal smearing), use the interactive tool to **remove** frames. If frames were dropped, use **Duplicate** to insert a copy. Then click **Export** and use result_sync in build_final_sync_df_merge_nearest() below.


In [ ]:
# Optional: run interactive sync tool; when satisfied click Export
result_sync = {}
display(interactive_sync_tool_bokeh(block, dfL_shifted, dfR_shifted, show_led=True, result_container=result_sync))
# If you exported, use result_sync['dfL'] and result_sync['dfR'] in build_final_sync_df_merge_nearest() instead of dfL_shifted, dfR_shifted


In [ ]:
# Optional: Hover inspection plot for detailed frame-by-frame inspection
# hover_inspect_eyes_bokeh(dfL_shifted, dfR_shifted)


In [ ]:
# Optional: Insert duplicate frames to correct for dropped frames
# Example usage (uncomment if needed):
# dfL_fix = insert_dup_by_pos(dfL_shifted, [3], duplicate="prev", leave_trailing_nan=True)
# dfR_fix = insert_dup_by_pos(dfR_shifted, [3], duplicate="prev", leave_trailing_nan=True)



## Step 7: Merge onto Arena Grid

Merge the corrected eye DataFrames onto the 60Hz arena grid using nearest-neighbor matching with tolerance. This creates the final synchronized dataframe (`final_sync_df`) used downstream.

In [11]:
# Merge onto the 60 Hz arena grid with nearest-with-tolerance (no resort of eye dfs)
final_df = build_final_sync_df_merge_nearest(
    block, dfL_shifted, dfR_shifted,
    target_fps=60.0,
    tol_frac=0.90,      # accept nearest within 90% of a 60 Hz tick; tune 0.7–1.2 if needed
    pre_shift_left=0,   # IMPORTANT: already pre-shifted
    pre_shift_right=0,
    export_csv=True
)

c:\Users\nimro\miniconda3\envs\eye_repo\lib\site-packages\pandas\core\base.py:666: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


[OK] Saved final sync CSV → D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\analysis\blocksync_df.csv
[INFO] Grid rows: 81,987 | tick ≈ 16.650 ms | tol=300 samp (tol_frac=0.90)
[INFO] Valid LEFT grid points: 79,160 | Valid RIGHT grid points: 79,152


## Step 8: Final Sync Verification
builds the brigtness grid but from final_df, if everything went according to plan you should get perfect alignment between eyes and with the LED driver off rising edges

In [12]:

fs = float(block.sample_rate)
sanity_plot_final_df(final_df, fs, show_led=True, block=block)

Loading BokehJS ...

## Step 9: Statistics-based verification and export


In [13]:
# dfL_s/dfR_s are the eye DFs you actually shifted (exact slider semantics)
stats = verify_final_df_against_sources(block, final_df, dfL_shifted, dfR_shifted, target_fps=60.0, tol_frac=0.9)


[VERIFY] Grid length=81987, tick≈16.650 ms, tol=300 samples
[VERIFY] Left  frame match: 100.000%   Left  values match: 100.000%
[VERIFY] Right frame match: 100.000%   Right values match: 100.000%


In [14]:
# Export the final synchronized dataframe
export_final_sync_df(block, final_df=final_df, overwrite=True)

[OK] Overwrote D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\analysis\blocksync_df.csv
[OK] Wrote D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\analysis\final_sync_df.csv
[OK] block.final_sync_df (and block.blocksync_df) set.


### Preprocessing of synchronized data from here (loads final_sync from folder if available)

In [15]:

for block in block_collection:
    load_final_sync_df(block)


[OK] Loaded final_sync_df.csv → block.final_sync_df (rows=81,987)


In [19]:
plot_sync_verification_with_electrophys(block, channel=1, to_browser=True)


LED TTL events: 23 rising edges, 23 falling edges
Extracting electrophysiology data from channel 1...
Recording duration: 1365.09 seconds (1365094.40 ms)
  Extracted chunk 10/137...
  Extracted chunk 20/137...
  Extracted chunk 30/137...
  Extracted chunk 40/137...
  Extracted chunk 50/137...
  Extracted chunk 60/137...
  Extracted chunk 70/137...
  Extracted chunk 80/137...
  Extracted chunk 90/137...
  Extracted chunk 100/137...
  Extracted chunk 110/137...
  Extracted chunk 120/137...
  Extracted chunk 130/137...
Extracted 27301888 samples from electrophysiology recording
Downsampled by factor 10 for visualization (2730189 points)

Plot created successfully!
  - Electrophysiology: 2730189 points (downsampled from 27301888)
  - Eye brightness: 81987 points
  - LED TTL rising edges (ON): 23 events (green solid lines)
  - LED TTL falling edges (OFF): 23 events (red dashed lines)


In [ ]:
for block in block_collection:
    # load relevant data
    block.handle_eye_videos()
    block.parse_open_ephys_events()
    block.handle_arena_files()
    block.get_eye_brightness_vectors()
    load_final_sync_df(block)
    # read deeplabcut annotations and construct ellipse parameters per video frame
    block.read_dlc_data(overwrite=False, export=True)

In [22]:
plot_led_off_events_viewer(block, channel=1, to_browser=True)


Extracting 23 windows of 500 ms around LED_driver falling edges (full-resolution ephys)...
Viewer ready: 23 events, full-resolution ephys in ±0.25s around each LED off.


In [ ]:
for block in block_collection:
    # cross-corr distance estimation between eye video frames to correct for jitter
    # (relatively slow, but only needs to be run once per block, will load from disk if available)
    block.get_jitter_reports(export=True, overwrite=False, remove_led_blinks=False, sort_on_loading=True)

In [ ]:
# perform jitter correction and remove led blinks
for block in block_collection:
    block.correct_jitter()
    block.find_led_blink_frames(plot=True)
    block.remove_led_blinks_from_eye_df(export=True)

In [ ]:
df_inds_to_remove_l, vid_inds_l = find_jittery_frames(block, 'left', max_distance=60, diff_threshold=5,
                                                      gap_to_bridge=24)
df_inds_to_remove_r, vid_inds_r = find_jittery_frames(block, 'right', max_distance=60, diff_threshold=5,
                                                      gap_to_bridge=24)

# These are verification plots for the jitter outlier removal functions:
# to verify, I want a bokeh explorable:
rdf = pd.DataFrame.from_dict(block.re_jitter_dict)
ldf = pd.DataFrame.from_dict(block.le_jitter_dict)

In [ ]:
# visualize right eye
uf.bokeh_plotter([rdf.top_correlation_dist], ['drift_distance'], peaks=vid_inds_r)

In [ ]:
# visualize left eye
uf.bokeh_plotter([ldf.top_correlation_dist], ['drift_distance'], peaks=vid_inds_l)

In [ ]:
# if you are happy with the results, remove the outliers
block.remove_eye_datapoints_based_on_video_frames('right', indices_to_nan=vid_inds_r)
block.remove_eye_datapoints_based_on_video_frames('left', indices_to_nan=vid_inds_l)

In [ ]:
# create the eye dataframes, integrating the cleaned le/re dataframes with the ellipse parameters
# this will create block.left_eye_data and block.right_eye_data
for block in block_collection:
    block.create_eye_data()

In [ ]:

# Export the final eye dataframes
for block in block_collection:
    export_eye_data_2d(block)